In [ ]:
#@title ⚡ Khởi Động OmniVoice GPU Worker & Đường Hầm Tunnel (Bấm Run)
#@markdown Chọn phương thức kết nối về máy local của bạn:
#@markdown - **ngrok**: Khuyên dùng! Dùng **Static Domain cố định**, cấu hình file `.env` 1 lần dùng mãi mãi.
#@markdown - **cloudflare**: Miễn phí không cần tài khoản, link ngẫu nhiên mỗi lần chạy.
TUNNEL_METHOD = "ngrok" #@param ["ngrok", "cloudflare"]
NGROK_AUTHTOKEN = "3JJU5ZC5zThuSkjjTF4SMZRum1r_5jStVjtjmpZG5CXdMsCih" #@param {type:"string"}
NGROK_STATIC_DOMAIN = "tipper-semantic-dropper.ngrok-free.dev" #@param {type:"string"}

import os
import sys
import time
import re
import subprocess
import urllib.request
import json
from IPython.display import display, HTML

# 1. Kiểm tra GPU CUDA
print("🔍 [1/5] Đang kiểm tra card đồ hoạ GPU...")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vui lòng vào menu: Runtime -> Change runtime type -> Chọn T4 GPU rồi bấm chạy lại ô này.")
print("✅ Nhận diện GPU thành công!\n" + gpu_check.stdout.split("\n")[8])

# 2. Cài đặt các thư viện cần thiết
print("\n📦 [2/5] Đang cài đặt thư viện cần thiết (OmniVoice, Accelerate, Transformers, FastAPI, Uvicorn, PyNgrok)...")
reqs = [
    "omnivoice", "accelerate>=0.34.0", "webdataset", "safetensors", "transformers>=4.40.0",
    "soundfile", "librosa", "pydub", "scipy", "av>=11.0.0", "faster-whisper>=1.0.0", "fastapi", "uvicorn[standard]",
    "python-multipart", "httpx", "pyngrok"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("✅ Đã cài đặt xong toàn bộ thư viện phụ thuộc!")

# 3. Thiết lập mã nguồn OmniVoice Colab Worker mới nhất
print("\n⚙️ [3/5] Đang đồng bộ mã nguồn OmniVoice Colab Worker mới nhất...")
worker_url = "https://raw.githubusercontent.com/tranvankha1989/TranslateVideo/main/backend/colab_worker.py"
loaded_from_github = False
try:
    urllib.request.urlretrieve(worker_url, "colab_worker.py")
    if os.path.exists("colab_worker.py") and os.path.getsize("colab_worker.py") > 1000:
        loaded_from_github = True
        print("✅ Đã đồng bộ thành công colab_worker.py mới nhất từ GitHub!")
except Exception as e:
    print(f"⚠️ Không thể tải trực tiếp từ GitHub ({e}), sử dụng mã nguồn tích hợp sẵn...")

if not loaded_from_github:
    import base64
    print("📝 Đang tạo colab_worker.py từ mã nguồn tích hợp...")
    WORKER_B64 = "..."
    with open("colab_worker.py", "wb") as f:
        f.write(base64.b64decode(WORKER_B64))
    print("✅ Đã ghi mã nguồn colab_worker.py thành công!")

# 4. Khởi động server ngầm và ghi log
print("\n🚀 [4/5] Đang khởi động OmniVoice GPU Worker & nạp mô hình vào VRAM...")
log_file = open("worker.log", "w", encoding="utf-8")
server_proc = subprocess.Popen([sys.executable, "-u", "colab_worker.py"], stdout=log_file, stderr=subprocess.STDOUT)

# Chờ server sẵn sàng bằng cách poll health endpoint (tối đa 90s)
is_ready = False
print("⏳ Đang tải trọng số mô hình k2-fsa/OmniVoice (vui lòng đợi ~30-60 giây)...", end="", flush=True)
for _ in range(45):
    if server_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/remote/health", timeout=2) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                is_ready = True
                print(f"\n✅ Server đã sẵn sàng! [{data.get('gpu_name')} - {data.get('vram_total_gb')}GB VRAM]")
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

if not is_ready:
    log_file.close()
    with open("worker.log", "r", encoding="utf-8") as f:
        err_output = f.read()
    raise RuntimeError(f"❌ Server không khởi động được! Chi tiết log lỗi:\n\n{err_output}")

# 5. Khởi chạy Đường hầm Tunnel (Ngrok Static Domain hoặc Cloudflare Tunnel)
tunnel_url = None

if TUNNEL_METHOD == "ngrok" and NGROK_AUTHTOKEN.strip():
    print("\n🌐 [5/5] Đang kết nối đường hầm bảo mật qua Ngrok...")
    try:
        from pyngrok import ngrok, conf
        ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
        domain_val = NGROK_STATIC_DOMAIN.strip() if NGROK_STATIC_DOMAIN.strip() else None
        if domain_val:
            print(f"🔗 Đang kết nối tới tên miền tĩnh Ngrok: {domain_val} ...")
            tunnel = ngrok.connect(8000, "http", domain=domain_val)
        else:
            print("🔗 Đang tạo đường hầm Ngrok ngẫu nhiên...")
            tunnel = ngrok.connect(8000, "http")
        tunnel_url = tunnel.public_url
    except Exception as ne:
        print(f"⚠️ Lỗi kết nối Ngrok ({ne}). Đang tự động chuyển sang Cloudflare Tunnel dự phòng...")

if not tunnel_url:
    print("\n🌐 [5/5] Đang kết nối đường hầm Cloudflare Tunnel dự phòng...")
    subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
    subprocess.run(["dpkg", "-i", "-E", "cloudflared-linux-amd64.deb"], capture_output=True)
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    start_time = time.time()
    while time.time() - start_time < 30:
        line = tunnel_proc.stdout.readline()
        if not line: continue
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    display(HTML('<h1><font color="green">✅ Kết nối thành công!</font></h1>'))
    is_static = bool(TUNNEL_METHOD == "ngrok" and NGROK_STATIC_DOMAIN.strip() and NGROK_STATIC_DOMAIN.strip() in tunnel_url)
    print("\n" + "="*70)
    print("🎉 GOOGLE COLAB GPU (T4 16GB) ĐÃ SẴN SÀNG HOẠT ĐỘNG!")
    print(f"👉 URL kết nối: {tunnel_url}")
    print("="*70)
    if is_static:
        print("\n✨ BẠN ĐANG DÙNG NGROK STATIC DOMAIN CỐ ĐỊNH!")
        print("💡 Chỉ cần cấu hình file backend/.env trên máy bạn 1 LẦN DUY NHẤT:")
        print("-"*70)
        print("USE_COLAB_GPU=true")
        print(f"COLAB_API_URL={tunnel_url}")
        print("-"*70)
        print("🚀 Những ngày sau chỉ cần mở Colab bấm Chạy, web sẽ TỰ ĐỘNG KẾT NỐI mà không bao giờ cần sửa lại .env!\n")
    else:
        print("\n📋 HÃY SAO CHÉP 2 DÒNG DƯỚI ĐÂY DÁN VÀO FILE backend/.env TRÊN MÁY BẠN:")
        print("-"*70)
        print("USE_COLAB_GPU=true")
        print(f"COLAB_API_URL={tunnel_url}")
        print("-"*70 + "\n")
    print("💡 Giữ tab Google Colab này mở trong suốt quá trình sử dụng hệ thống.\n")
    try:
        if TUNNEL_METHOD == "ngrok" and NGROK_AUTHTOKEN.strip() and "ngrok" in tunnel_url:
            from pyngrok import ngrok
            ngrok.get_ngrok_process().proc.wait()
        else:
            tunnel_proc.wait()
    except KeyboardInterrupt:
        print("🛑 Đã dừng worker.")
else:
    print("⚠️ Không tìm thấy URL kết nối. Vui lòng thử khởi động lại ô này.")

🔍 [1/5] Đang kiểm tra card đồ hoạ GPU...


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'